# Conditional inference — v3

Demonstrates real-time-label conditioning on `MixedTypeMultiTargetScorer`:

- Pair the scorer with `ConditionalJointMultiTargetMLPEstimator` (or `…TransformerEstimator`).
- At inference, supply `OBSERVED_<suffix>` columns where ground truth is known per row.
- NaN per cell means "not observed, predict from features." Multilabel group members must mask together per row.

Extends the vanilla joint-families demo with the conditional path. See [joint_families.ipynb](joint_families.ipynb) for the v2 baseline (without conditioning), [independent.ipynb](independent.ipynb) for the XGBoost/LightGBM-based per-target alternative, and [decision-rule.md](../../docs/user-guide/decision-rule.md) for when to use conditional inference.

In [1]:
# ruff: noqa: E402  (thread-pool env vars must be set before imports)
# macOS BLAS/OpenMP collision guard — must be set BEFORE numpy/torch import.
import os

for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

# Pre-import torch upfront. This notebook is torch-only (no lightgbm/
# xgboost) so the v2-quickstart-style cross-backend collision isn't a
# concern here; the env vars above are belt-and-suspenders.
import warnings

import torch  # noqa: F401, E402

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from skrec.constants import OBSERVED_PREFIX, USER_ID_NAME
from skrec.estimator.classification import (
    ConditionalJointMultiTargetMLPEstimator,
    ConditionalMultiTargetEstimator,
)
from skrec.orchestrator import capability_matrix
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.scorer.mixed_type_multi_target import (
    MixedTypeMultiTargetScorer,
    TargetGroupSpec,
    TargetType,
)

print("scorer_supports_observed_conditioning:", capability_matrix()["scorer_supports_observed_conditioning"])

scorer_supports_observed_conditioning: ('mixed_type_multi_target',)


## 1. Synthetic correlated-targets data

ITEM_a and ITEM_b are strongly correlated through a shared latent feature. ITEM_revenue is independent. ITEM_email_open + ITEM_app_open form a multilabel group.

In [2]:
rng = np.random.default_rng(42)
n = 600
X = pd.DataFrame(rng.normal(size=(n, 4)), columns=[f"feat_{i}" for i in range(4)])

# Correlated binary targets through a shared latent.
latent = X["feat_0"] + 0.5 * X["feat_1"]
y_a = (latent > 0).astype(int).to_numpy()
y_b = (latent + 0.2 * rng.normal(size=n) > 0).astype(int).to_numpy()
y_revenue = (2.5 * X["feat_2"] + 0.1 * rng.normal(size=n)).to_numpy()
y_engagement = np.column_stack(
    [
        (X["feat_3"] > 0).astype(int).to_numpy(),
        (X["feat_3"] + 0.3 * rng.normal(size=n) > 0).astype(int).to_numpy(),
    ]
)

target_specs = {
    "ITEM_a": TargetType.BINARY,
    "ITEM_b": TargetType.BINARY,
    "ITEM_revenue": TargetType.REGRESSION,
    "engagement": TargetGroupSpec(
        type=TargetType.MULTILABEL,
        columns=["ITEM_email_open", "ITEM_app_open"],
    ),
}
y = {
    "ITEM_a": y_a,
    "ITEM_b": y_b,
    "ITEM_revenue": y_revenue,
    "engagement": y_engagement,
}
split = 500
X_train, X_valid = X.iloc[:split], X.iloc[split:].reset_index(drop=True)
y_train = {k: v[:split] for k, v in y.items()}
y_valid = {k: v[split:] for k, v in y.items()}
print("train:", X_train.shape, "valid:", X_valid.shape)

train: (500, 4) valid: (100, 4)


## 2. Train a conditional joint MLP

`mask_prob=0.5` means each (row, target) draw is masked 50% of the time at training. The model learns to predict masked targets given the unmasked ones via the label channel.

In [3]:
estimator = ConditionalJointMultiTargetMLPEstimator(
    target_specs=target_specs,
    params={
        "epochs": 15,
        "hidden_dim": 32,
        "num_layers": 2,
        "batch_size": 64,
        "mask_prob": 0.5,
        "label_embedding_dim": 8,
        "seed": 0,
    },
)
estimator.fit(X_train, y_train)

assert isinstance(estimator, ConditionalMultiTargetEstimator)
print(
    "Trained.",
    "\nImplements ConditionalMultiTargetEstimator Protocol:",
    isinstance(estimator, ConditionalMultiTargetEstimator),
)

2026-05-26 04:49:34,810 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [1/15] - Conditional Train Loss: 0.7827


2026-05-26 04:49:34,816 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [2/15] - Conditional Train Loss: 0.7273


2026-05-26 04:49:34,822 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [3/15] - Conditional Train Loss: 0.7544


2026-05-26 04:49:34,827 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [4/15] - Conditional Train Loss: 0.7321


2026-05-26 04:49:34,832 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [5/15] - Conditional Train Loss: 0.6960


...

2026-05-26 04:49:34,869 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [12/15] - Conditional Train Loss: 0.4409


2026-05-26 04:49:34,874 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [13/15] - Conditional Train Loss: 0.4097


2026-05-26 04:49:34,879 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [14/15] - Conditional Train Loss: 0.3954


2026-05-26 04:49:34,884 - skrec.estimator.classification._conditional_joint_multi_target_base - INFO Epoch [15/15] - Conditional Train Loss: 0.3755


Trained. 
Implements ConditionalMultiTargetEstimator Protocol: True


## 3. Vanilla (no conditioning) vs. conditional (observe ITEM_a)

Since ITEM_a and ITEM_b are correlated, observing ITEM_a should shift the model's prediction for ITEM_b on the same row.

In [4]:
# Build a wide inference DataFrame for the valid slice.
scorer = MixedTypeMultiTargetScorer(estimator=estimator, target_specs=target_specs)
recommender = RankingRecommender(scorer=scorer)

inf_df = X_valid.copy()
inf_df.insert(0, USER_ID_NAME, [f"u_{i}" for i in range(len(X_valid))])

# Vanilla path: no OBSERVED_* columns, predictions from features alone.
out_vanilla = scorer.score_items(interactions=inf_df)
p_b_vanilla = out_vanilla["ITEM_b_1"].to_numpy()

# Conditional path: observe ITEM_a = 1 for every row.
inf_df_obs = inf_df.copy()
inf_df_obs[f"{OBSERVED_PREFIX}a"] = 1.0
out_observed_a_pos = scorer.score_items(interactions=inf_df_obs)
p_b_observed = out_observed_a_pos["ITEM_b_1"].to_numpy()

mean_shift = float(np.mean(p_b_observed - p_b_vanilla))
print(f"Mean shift in P(ITEM_b=1) after observing ITEM_a=1: {mean_shift:+.4f}")
print("(Sign should be positive — observing a positive correlated target")
print(" raises the probability of the other.)")

Mean shift in P(ITEM_b=1) after observing ITEM_a=1: +0.1413
(Sign should be positive — observing a positive correlated target
 raises the probability of the other.)


## 4. Multilabel group: members must mask together per row

v3 locked decision #4: within a multilabel group, all members are observed together OR all NaN together per row. Partial-group observation raises a clean error at the scorer's validator.

In [5]:
# Legal: observe BOTH members or NEITHER per row.
legal = inf_df.copy()
legal[f"{OBSERVED_PREFIX}email_open"] = 1.0
legal[f"{OBSERVED_PREFIX}app_open"] = 0.0
_ = scorer.score_items(interactions=legal)
print("Legal: both members observed →", scorer.score_items(interactions=legal).shape)

# Illegal: observe one member, NaN the other for the same row.
illegal = inf_df.copy()
illegal[f"{OBSERVED_PREFIX}email_open"] = 1.0
illegal[f"{OBSERVED_PREFIX}app_open"] = np.nan
try:
    scorer.score_items(interactions=illegal)
except ValueError as e:
    print("Illegal: partial-group observation →", str(e).split(".")[0])

Legal: both members observed → (100, 9)
Illegal: partial-group observation → Multilabel group 'engagement' has partial-group observation in row(s) [0, 1, 2, 3, 4] (showing up to 5)


## 5. Vanilla estimators still reject OBSERVED_*

Switching back to a non-conditional estimator preserves the v2 behavior: any `OBSERVED_*` column at inference raises with a pointer to the conditional family.

In [6]:
from skrec.estimator.classification import JointMultiTargetMLPEstimator

vanilla = JointMultiTargetMLPEstimator(
    target_specs=target_specs,
    params={"epochs": 2, "hidden_dim": 16, "num_layers": 1},
)
vanilla.fit(X_train, y_train)
vanilla_scorer = MixedTypeMultiTargetScorer(estimator=vanilla, target_specs=target_specs)

obs_df = inf_df.copy()
obs_df[f"{OBSERVED_PREFIX}a"] = 1.0
try:
    vanilla_scorer.score_items(interactions=obs_df)
except NotImplementedError as e:
    print("Vanilla rejects OBSERVED_* with:", str(e).split(".")[0])

2026-05-26 04:49:34,902 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [1/2] - Train Loss: 0.7930


2026-05-26 04:49:34,903 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [2/2] - Train Loss: 0.7867


Vanilla rejects OBSERVED_* with: OBSERVED_* columns require a ConditionalMultiTargetEstimator (e


## 6. Schema preservation through `recommend_online`

`OBSERVED_*` columns are auto-preserved through `interactions_schema.apply()` even when the client schema doesn't declare them (via `BaseScorer.preserved_inference_columns()`). So `recommend_online` works without per-deployment schema changes.

In [7]:
print("Scorer-side preserved-columns hook returns:")
print(" ", sorted(scorer.preserved_inference_columns()))
print("\nMapping: ITEM_<suffix> ↔ OBSERVED_<suffix>")
print(" Targets:", sorted(scorer._fanned_out_target_columns))

Scorer-side preserved-columns hook returns:
  ['OBSERVED_a', 'OBSERVED_app_open', 'OBSERVED_b', 'OBSERVED_email_open', 'OBSERVED_revenue']

Mapping: ITEM_<suffix> ↔ OBSERVED_<suffix>
 Targets: ['ITEM_a', 'ITEM_app_open', 'ITEM_b', 'ITEM_email_open', 'ITEM_revenue']


## 7. Per-target `evaluate()` with OBSERVED-aware dispatch

The v3 conditional path doesn't change the evaluation contract — `RankingRecommender.evaluate()` still returns `Dict[str, float]` with per-`TargetType` metric dispatch. The OBSERVED-aware dispatch in the scorer routes `predict_proba_dict` through `predict_with_observed` automatically when `OBSERVED_*` columns are in the interactions frame, so evaluation honors conditioning end-to-end.

In [8]:
from skrec.evaluator.datatypes import RecommenderEvaluatorType
from skrec.metrics.datatypes import RecommenderMetricType

# Wide-format logged_rewards (multilabel members fanned out per column).
logged = pd.DataFrame(
    {
        "ITEM_a": y_valid["ITEM_a"],
        "ITEM_b": y_valid["ITEM_b"],
        "ITEM_revenue": y_valid["ITEM_revenue"],
        "ITEM_email_open": y_valid["engagement"][:, 0],
        "ITEM_app_open": y_valid["engagement"][:, 1],
    }
)
per_target_metrics = {
    "ITEM_a": RecommenderMetricType.ROC_AUC,
    "ITEM_b": RecommenderMetricType.ROC_AUC,
    "ITEM_revenue": RecommenderMetricType.MAE,
    "ITEM_email_open": RecommenderMetricType.ROC_AUC,
    "ITEM_app_open": RecommenderMetricType.ROC_AUC,
}

# Vanilla evaluate — no OBSERVED_* columns. Equivalent to unconditional path.
out_vanilla = recommender.evaluate(
    eval_type=RecommenderEvaluatorType.SIMPLE,
    metric_type=per_target_metrics,
    eval_top_k=10,
    score_items_kwargs={"interactions": inf_df},
    eval_kwargs={"logged_rewards": logged},
)

# Conditional evaluate — observe true ITEM_a per row. ITEM_b metric should
# improve because the model can leverage the observed correlated target.
inf_df_with_obs_a = inf_df.copy()
inf_df_with_obs_a[f"{OBSERVED_PREFIX}a"] = y_valid["ITEM_a"].astype(float)
out_with_obs_a = recommender.evaluate(
    eval_type=RecommenderEvaluatorType.SIMPLE,
    metric_type=per_target_metrics,
    eval_top_k=10,
    score_items_kwargs={"interactions": inf_df_with_obs_a},
    eval_kwargs={"logged_rewards": logged},
)

print("Evaluate — vanilla vs OBSERVED_a:")
print(f"{'target':<20}{'vanilla':>12}{'observe(ITEM_a)':>20}{'delta':>10}")
for k in per_target_metrics:
    v_van = out_vanilla[k]
    v_obs = out_with_obs_a[k]
    print(f"{k:<20}{v_van:>12.4f}{v_obs:>20.4f}{v_obs - v_van:>+10.4f}")

Evaluate — vanilla vs OBSERVED_a:
target                   vanilla     observe(ITEM_a)     delta
ITEM_a                    0.9838              0.9968   +0.0129
ITEM_b                    0.9767              0.9800   +0.0033
ITEM_revenue              0.3029              0.3010   -0.0019
ITEM_email_open           0.9464              0.9520   +0.0056
ITEM_app_open             0.8260              0.8288   +0.0028


### `score_per_target` — user-supplied callables on the conditional model

Same escape-hatch as the v2 vanilla scorer: pass sklearn metrics keyed by `TargetType` (or target name for per-target overrides). For conditional inference, supply `OBSERVED_*` columns in `interactions` and the scorer routes through `predict_with_observed`.

In [9]:
from sklearn.metrics import log_loss, mean_absolute_percentage_error

user_metrics = scorer.score_per_target(
    interactions=inf_df_with_obs_a,  # OBSERVED_a present — conditional path
    y_true=logged,
    metric_callables={
        TargetType.BINARY: lambda yt, p: float(log_loss(yt, p[:, 1], labels=[0, 1])),
        TargetType.REGRESSION: lambda yt, p: float(mean_absolute_percentage_error(np.clip(np.abs(yt), 1e-6, None), p)),
    },
)
print("User-supplied metrics (sklearn callables) under OBSERVED_a conditioning:")
for k, v in user_metrics.items():
    print(f"  {k:20s} = {v:.4f}")

User-supplied metrics (sklearn callables) under OBSERVED_a conditioning:
  ITEM_a               = 0.3836
  ITEM_b               = 0.3214
  ITEM_revenue         = 1.0535
  ITEM_email_open      = 0.5853
  ITEM_app_open        = 0.5827


## Where to go next

- **Joint families (vanilla, PyTorch)**: [joint_families.ipynb](joint_families.ipynb) — baseline without conditioning
- **Independent (XGBoost/LightGBM)**: [independent.ipynb](independent.ipynb) — per-target sub-estimators with tree-based learners (no conditioning support)
- **v3 plan**: [mixed_type_multi_target_plan_v3.md](../../mixed_type_multi_target_plan_v3.md) — locked design decisions, label encoding, mandatory leakage gate
- **Decision rule**: [docs/user-guide/decision-rule.md](../../docs/user-guide/decision-rule.md) — when to use conditional inference